# Experiment 9D: Sample Efficiency Curves

Map how BERT and ModernBERT scale with training data size.
Train at 500, 1K, 2K, 4K, 8K, 13K samples x 3 seeds each.
If gap shrinks at larger scales, ModernBERT needs more data.
If gap is constant, pre-training alignment is the cause.

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn peft accelerate transformers sentencepiece protobuf

In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, training_args
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import json
import gc
import matplotlib.pyplot as plt

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
FPB_SOURCE = 5
SAMPLE_SIZES = [500, 1000, 2000, 4000, 8000, 13000]
SEEDS = [42, 123, 456]

label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

all_texts = ds["train"]["text"] + ds["validation"]["text"]
all_labels_str = ds["train"]["label"] + ds["validation"]["label"]
all_labels = [label_dict[l] for l in all_labels_str]

print(f"Total available training samples: {len(all_texts)}")
print(f"Label distribution: {pd.Series(all_labels).value_counts().sort_index().to_dict()}")

def load_fpb_file(agree_level):
    from huggingface_hub import hf_hub_download
    import zipfile, os
    path = hf_hub_download("financial_phrasebank", "data/FinancialPhraseBank-v1.0.zip", repo_type="dataset")
    extract_dir = "/tmp/fpb_data"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(extract_dir)
    filename = {"50agree": "Sentences_50Agree.txt", "allagree": "Sentences_AllAgree.txt"}[agree_level]
    filepath = os.path.join(extract_dir, "FinancialPhraseBank-v1.0", filename)
    sentences, labels = [], []
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            at_idx = line.rfind("@")
            if at_idx == -1:
                continue
            label_str = line[at_idx+1:].strip().lower()
            if label_str in label_map:
                sentences.append(line[:at_idx].strip())
                labels.append(label_map[label_str])
    return {"sentence": sentences, "label": labels}

def load_fpb_dataset(agree_level="50agree"):
    try:
        config = f"sentences_{agree_level}"
        return load_dataset("financial_phrasebank", config, trust_remote_code=True)["train"]
    except Exception:
        return load_fpb_file(agree_level)

fpb_50 = load_fpb_dataset("50agree")
fpb_all = load_fpb_dataset("allagree")
print(f"FPB 50agree: {len(fpb_50['sentence']):,}  |  FPB allAgree: {len(fpb_all['sentence']):,}")

In [ ]:
label_dict = {"NEUTRAL/MIXED": 1, "NEGATIVE": 0, "POSITIVE": 2}

ds = load_dataset("neoyipeng/financial_reasoning_aggregated")
ds = ds.filter(lambda x: x["task"] == "sentiment")
ds = ds.filter(lambda x: x["source"] != FPB_SOURCE)

all_texts = ds["train"]["text"] + ds["validation"]["text"]
all_labels_str = ds["train"]["label"] + ds["validation"]["label"]
all_labels = [label_dict[l] for l in all_labels_str]

print(f"Total available training samples: {len(all_texts)}")
print(f"Label distribution: {pd.Series(all_labels).value_counts().sort_index().to_dict()}")

fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
fpb_all = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb_50):,}  |  FPB allAgree: {len(fpb_all):,}")

## 2. Evaluation Helper

In [ ]:
def evaluate_on_fpb(model, tokenizer, fpb_dataset, batch_size=32):
    fpb_texts = fpb_dataset["sentence"]
    fpb_labels = fpb_dataset["label"]
    all_preds = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(fpb_texts), batch_size):
            batch = fpb_texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.cuda() for k, v in inputs.items()}
            logits = model(**inputs).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
    y_true = np.array(fpb_labels)
    y_pred = np.array(all_preds)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

## 3. Training Function

In [ ]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, lbls):
        self.encodings = encodings
        self.labels = lbls
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

def train_and_eval(model_name, lora_targets, train_texts_sub, train_labels_sub, val_texts_sub, val_labels_sub, seed, run_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_CLASSES)

    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=lora_targets,
        lora_dropout=0.05, bias="none",
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_config)
    model.gradient_checkpointing_enable()

    train_enc = tokenizer(train_texts_sub, truncation=True, padding=True, max_length=128)
    val_enc = tokenizer(val_texts_sub, truncation=True, padding=True, max_length=128)
    train_ds = SimpleDataset(train_enc, train_labels_sub)
    val_ds = SimpleDataset(val_enc, val_labels_sub)

    args = TrainingArguments(
        output_dir=f"efficiency_output/{run_name}",
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.001,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=50,
        seed=seed,
        fp16=True,
        report_to="none",
        save_total_limit=1,
    )

    def compute_metrics(eval_pred):
        preds = np.argmax(eval_pred.predictions, axis=-1)
        return {"accuracy": accuracy_score(eval_pred.label_ids, preds)}

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    model = model.cuda().eval()

    res_50 = evaluate_on_fpb(model, tokenizer, fpb_50)
    res_all = evaluate_on_fpb(model, tokenizer, fpb_all)

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return res_50["accuracy"], res_all["accuracy"], res_50["macro_f1"]

## 4. Run All Experiments

In [ ]:
MODELS_CONFIG = {
    "bert-base": {
        "name": "bert-base-uncased",
        "targets": ["query", "key", "value", "dense"],
    },
    "modernbert-base": {
        "name": "answerdotai/ModernBERT-base",
        "targets": ["Wqkv", "out_proj", "Wi", "Wo"],
    },
}

results = []

for model_key, cfg in MODELS_CONFIG.items():
    for n_samples in SAMPLE_SIZES:
        for seed in SEEDS:
            n_actual = min(n_samples, len(all_texts))
            if n_actual < len(all_texts):
                sub_texts, _, sub_labels, _ = train_test_split(
                    all_texts, all_labels,
                    train_size=n_actual,
                    stratify=all_labels,
                    random_state=seed,
                )
            else:
                sub_texts = all_texts
                sub_labels = all_labels

            train_t, val_t, train_l, val_l = train_test_split(
                sub_texts, sub_labels,
                test_size=0.1,
                stratify=sub_labels,
                random_state=seed,
            )

            run_name = f"{model_key}_n{n_actual}_s{seed}"
            print(f"\n{'='*60}")
            print(f"{run_name} (train={len(train_t)}, val={len(val_t)})")
            print(f"{'='*60}")

            acc_50, acc_all, f1_50 = train_and_eval(
                cfg["name"], cfg["targets"],
                train_t, train_l, val_t, val_l,
                seed, run_name
            )

            result = {
                "model": model_key,
                "n_samples": n_actual,
                "seed": seed,
                "fpb_50agree_acc": acc_50,
                "fpb_allagree_acc": acc_all,
                "fpb_50agree_f1": f1_50,
            }
            results.append(result)
            print(f"  FPB 50agree: {acc_50:.4f}  |  allAgree: {acc_all:.4f}")

with open("results_09d.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nAll results saved to results_09d.json")

## 5. Plot Scaling Curves

In [ ]:
df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric, title in [
    (axes[0], "fpb_50agree_acc", "FPB sentences_50agree"),
    (axes[1], "fpb_allagree_acc", "FPB sentences_allAgree"),
]:
    for model_key, color in [("bert-base", "#4CAF50"), ("modernbert-base", "#2196F3")]:
        subset = df[df.model == model_key]
        grouped = subset.groupby("n_samples")[metric]
        means = grouped.mean()
        stds = grouped.std()
        ax.errorbar(means.index, means.values * 100, yerr=stds.values * 100,
                    label=model_key, marker='o', capsize=4, color=color, linewidth=2)

    ax.set_xlabel("Training Samples")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')

plt.suptitle("Exp 9D: Sample Efficiency — BERT vs ModernBERT", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("09d_sample_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Gap Analysis

In [ ]:
print("\nGap (BERT - ModernBERT) at each scale:")
print(f"{'N':>6s}  {'BERT 50%':>10s}  {'MB 50%':>10s}  {'Gap':>8s}  {'BERT all%':>10s}  {'MB all%':>10s}  {'Gap':>8s}")
print("-" * 72)

for n in SAMPLE_SIZES:
    bert_50 = df[(df.model == "bert-base") & (df.n_samples == n)]["fpb_50agree_acc"].mean()
    mb_50 = df[(df.model == "modernbert-base") & (df.n_samples == n)]["fpb_50agree_acc"].mean()
    bert_all = df[(df.model == "bert-base") & (df.n_samples == n)]["fpb_allagree_acc"].mean()
    mb_all = df[(df.model == "modernbert-base") & (df.n_samples == n)]["fpb_allagree_acc"].mean()
    print(f"{n:>6d}  {bert_50:>9.2%}  {mb_50:>9.2%}  {(bert_50-mb_50)*100:>+7.2f}pp  {bert_all:>9.2%}  {mb_all:>9.2%}  {(bert_all-mb_all)*100:>+7.2f}pp")

print("\nInterpretation:")
gaps = []
for n in SAMPLE_SIZES:
    bert_50 = df[(df.model == "bert-base") & (df.n_samples == n)]["fpb_50agree_acc"].mean()
    mb_50 = df[(df.model == "modernbert-base") & (df.n_samples == n)]["fpb_50agree_acc"].mean()
    gaps.append(bert_50 - mb_50)

if gaps[-1] < gaps[0] * 0.5:
    print("Gap SHRINKS with more data -> ModernBERT needs more data to compensate.")
elif gaps[-1] > gaps[0] * 0.8:
    print("Gap PERSISTS across scales -> Pre-training alignment is the fundamental cause.")
else:
    print("Gap partially shrinks -> Both data scale and pre-training contribute.")